# Prompt engineering and LangChain prompt templates!

### Prompting techniques
- zero-shot 
- one-shot 
- few-shot learning 
- chain-of-thought reasoning 
- self-consistency prompting.

In [ ]:
# Suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings("ignore")

# OpenAI + LangChain imports
from langchain_openai import ChatOpenAI

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence
from langchain_core.messages import HumanMessage, SystemMessage

from langchain.chains import LLMChain  # still supported - It's deprecated but still works

from IPython.display import Markdown, display


In [5]:
llm = ChatOpenAI(
    model="gpt-4.1-nano",
    temperature=0.2
)
llm.invoke([HumanMessage(content="Hello")]) # invoke an llm with a human message


AIMessage(content='Hello! How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'finish_reason': 'stop', 'logprobs': None}, id='run-11129522-f395-4e9b-a1eb-a3dcd8a96039-0', usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17})

In [ ]:
def llm_model(prompt_txt, params=None):
    default_params = {
        "max_tokens": 256,
        "temperature": 0.5, # Higher temperature increases the probability of selecting lower-probability tokens. - In my opinion, only use this to control diversity of outputs and not top_p
        "top_p": 0.2, # to control the randomness and diversity of the generated text - It is generally recommended to adjust one or the other, but not both simultaneously,
        "timeout":None,
    }

    if params:
        default_params.update(params)

    llm = ChatOpenAI(
        model="gpt-4.1-nano",
        **default_params
    )

    response = llm.invoke([HumanMessage(content=prompt_txt)])
    return response.content


## Basic prompt


In [16]:
params = {
    "max_tokens": 128,
    "temperature": 1,
    "top_p": 0.2,
}

prompts = [
    "The wind is ",
    "The future of artificial intelligence is",
    # "Once upon a time in a distant galaxy",
    # "The benefits of sustainable energy include"
]

for prompt in prompts:
    response = llm_model(prompt, params)
    print(f"prompt: {prompt}\n")
    print(f"response : {response}\n")

prompt: The wind is 

response : The wind is the movement of air from high-pressure areas to low-pressure areas in the Earth's atmosphere. It plays a crucial role in weather patterns, climate, and the distribution of heat and moisture around the planet. Wind can vary in speed and direction, ranging from gentle breezes to strong storms.

prompt: The future of artificial intelligence is

response : The future of artificial intelligence (AI) holds immense potential to transform various aspects of society, technology, and daily life. As AI continues to advance, we can anticipate several key developments:

1. **Enhanced Automation and Productivity:** AI will automate more complex tasks across industries such as manufacturing, healthcare, finance, and transportation, leading to increased efficiency and new opportunities for innovation.

2. **Personalized Experiences:** From personalized medicine to tailored education and entertainment, AI will enable highly customized experiences that cater 

## Zero-shot prompt
- **Zero-shot** prompting is a technique where the model performs a task without any examples or prior specific training on that task. 
- **Zero-shot** prompts typically include clear instructions about what the model should do, allowing it to leverage its pre-trained knowledge effectively.

In [19]:
prompt = """Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'
"""
response = llm_model(prompt)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

translation_prompt = """
Translate the following English phrase into Urdu.

English: "I would like to order a coffee with milk and two sugars, please."
"""

response = llm_model(translation_prompt)
print(f"translation_prompt: {translation_prompt}\n")
print(f"response : {response}\n")

prompt: Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'


response : False

translation_prompt: 
Translate the following English phrase into Urdu.

English: "I would like to order a coffee with milk and two sugars, please."


response : میں ایک کافی دودھ کے ساتھ اور دو چینی کے ساتھ آرڈر کرنا چاہوں گا، براہ مہربانی۔



## One-shot prompt
- **One-shot prompting** provides the model with a single example of the task before asking it to perform a similar task. 
- This technique gives the model a pattern to follow, improving its understanding of the desired output format and style.

In [24]:
params = {
    "max_tokens": 50,
    "temperature": 0.1,
}

prompt = """Here is an example of translating a sentence from English to Roman Urdu:

            English: “How is the weather today?”
            Roman: Aaj mosam kesa he?”
            
            Now, translate the following sentence from English to Roman:
            
            English: “Where is the nearest supermarket?”
            
"""
response = llm_model(prompt, params=params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here is an example of translating a sentence from English to Roman Urdu:

            English: “How is the weather today?”
            Roman: Aaj mosam kesa he?”

            Now, translate the following sentence from English to Roman:

            English: “Where is the nearest supermarket?”



response : Qareeb tareen supermarket kahan he?



In [25]:
# 1. One-shot prompt for formal email writing
formal_email_prompt = """
                            Write a format email. You can check email format from below example:
                            
                            Subject: Remote MERN Position:
                            Assalam-O-Alaikum Hiring Manager,
                            I see that you need a MERN developer. I have 5 years of experience and have worked on diverse projects. 
                            I have attached my resume for your consideration.
                            Best Regards,

                            Now, write an email to my manager at the software house for promotion.
"""

# 3. One-shot prompt for keyword extraction
keyword_extraction_prompt = """
Here is an example of extracting keywords from a sentence:

Sentence: "Cloud computing offers businesses flexibility, scalability, and cost-efficiency for their IT infrastructure needs."
Keywords: cloud computing, flexibility, scalability, cost-efficiency, IT infrastructure

---

Now, please extract the main keywords from the following sentence:

Sentence: "Sustainable agriculture practices focus on biodiversity, soil health, water conservation, and reducing chemical inputs."
Keywords:
"""

responses = {}
responses["formal_email"] = llm_model(formal_email_prompt)
responses["keyword_extraction"] = llm_model(keyword_extraction_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()


=== FORMAL_EMAIL RESPONSE ===
Subject: Request for Promotion

Assalam-O-Alaikum [Manager's Name],

I hope this message finds you well. I would like to kindly request your consideration for a promotion. Over the past [duration], I have diligently worked on various projects, contributed to team success, and taken on additional responsibilities to support our goals.

I am eager to continue growing professionally and believe that a promotion would enable me to contribute even more effectively to our team and company.

Thank you for your time and consideration. I look forward to your feedback.

Best Regards,  
[Your Name]

=== KEYWORD_EXTRACTION RESPONSE ===
biodiversity, soil health, water conservation, chemical inputs



## Few-shot prompt
- Few-shot prompting extends the one-shot approach by providing multiple examples (typically 2-5) before asking the model to perform the task. 
- These examples establish a clearer pattern and context, helping the model better understand the expected output format, style, and reasoning. 
- This technique is particularly effective for complex tasks where a single example might not convey all the nuances.

In [29]:
#parameters: Set `max_tokens` to 10, which constrains the model to generate brief responses

params = {
    "max_tokens": 10,
}

prompt = """Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            Emotion:
            
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy

            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration

            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness

            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            Emotion:



response : Emotion: Fear



## Chain-of-thought (CoT) prompt
- Chain-of-thought (CoT) prompting encourages the model to break down complex problems into step-by-step reasoning before arriving at a final answer. By explicitly showing or requesting intermediate steps. 
- This technique improves the model's problem-solving abilities and reduces errors in tasks requiring multi-step reasoning. 
- CoT is particularly effective for mathematical problems, logical reasoning, and complex decision-making tasks.

In [30]:
params = {
    "max_tokens": 512,
    "temperature": 0.5,
}

prompt = """Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

prompt: Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation



response : Let's break down the problem step by step:

1. **Initial number of apples:**  
   The store started with **22 apples**.

2. **Apples sold today:**  
   They sold **15 apples**.  
   To find out how many apples remain after the sale, subtract the sold apples from the initial amount:  
   **22 - 15 = 7 apples** remaining.

3. **New delivery of apples:**  
   The store received **8 new apples**.  
   Add these to the remaining apples:  
   **7 + 8 = 15 apples**.

**Final answer:**  
There are **15 apples** now.



## Self-consistency
- Self-consistency is an advanced technique in which the model generates multiple independent solutions or answers to the same problem, then evaluates these different approaches to determine the most consistent or reliable result. 
- This method enhances accuracy by leveraging the model's ability to approach problems from different angles and identify the most robust solution through comparison and verification.

In [36]:
params = {
    "max_tokens": 512,
}

prompt = """When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {display(Markdown(response))}\n")

prompt: When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.





Let's analyze the problem step-by-step:

**Given:**
- When you were 6 years old, your sister was half your age.
- Now, you are 70 years old.
- Find your sister's current age.

---

### Calculation 1: Direct Age Difference Method

**Step 1:** Find your sister's age when you were 6.

- When you were 6, your sister was half of that:  
  \( \frac{1}{2} \times 6 = 3 \) years old.

**Step 2:** Determine the age difference.

- The age difference between you and your sister is constant:  
  \( 6 - 3 = 3 \) years.

**Step 3:** Find your sister's current age.

- Since you are now 70, your sister's age is:  
  \( 70 - 3 = 67 \).

**Result:** **Your sister is 67 years old.**

---

### Calculation 2: Using the Age Difference and Current Age

**Step 1:** Age difference is the same as in the past: 3 years.

**Step 2:** Your current age is 70.

**Step 3:** Subtract the age difference to find your sister's age:

- \( 70 - 3 = 67 \).

**Result:** **Your sister is 67 years old.**

---

### Calculation 3: Cross-Verification with a Variable

Let:

- \( S \) = your sister's current age.
- \( A \) = your current age = 70.

**Step 1:** When you were 6, your sister was half your age:

- Sister's age at that time: 3 years old.
- Age difference: \( 6 - 3 = 3 \).

**Step 2:** Since the age difference is constant:

- \( S = A - 3 = 70 - 3 = 67 \).

**Result:** **Your sister is 67 years old.**

---

### **Most Consistent Result:**

All three independent calculations agree that **your sister is 67 years old**.

---

**Final answer: \(\boxed{67}\) years old.**

response : None

